# Document → Structured JSON with Eden AI

Drop a receipt, resume, or invoice. Four vision LLMs extract a JSON object matching the same schema, side-by-side. See who follows the schema, who hallucinates fields, and who refuses to guess.

Same API, same key, same multimodal message shape as the [Vision Arena](vision_arena.ipynb) — what changes is the prompt and the post-call JSON validation.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var or in `.env`).

In [ ]:
%pip install --quiet aiohttp ipywidgets nest_asyncio python-dotenv requests

## 1. Configuration

Four vision-capable models, three pre-defined schemas. Add/remove models freely — every entry below has `image` in its `input_modalities` per the Eden AI catalog. Schemas are plain Python dicts; bring your own.

In [ ]:
import base64
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_URL = "https://api.edenai.run/v3/llm/chat/completions"

MODELS = [
    {"label": "Claude",  "model": "anthropic/claude-sonnet-4-5"},
    {"label": "GPT-4o",  "model": "openai/gpt-4o"},
    {"label": "Gemini",  "model": "google/gemini-2.5-flash"},
    {"label": "Pixtral", "model": "mistral/pixtral-large-latest"},
]

SCHEMAS = {
    "Receipt": {
        "description": "Extract structured data from a retail or restaurant receipt.",
        "schema": {
            "type": "object",
            "properties": {
                "vendor":   {"type": ["string", "null"]},
                "date":     {"type": ["string", "null"]},
                "currency": {"type": ["string", "null"]},
                "total":    {"type": ["number", "null"]},
                "items":    {"type": "array", "items": {
                    "type": "object",
                    "properties": {
                        "name":   {"type": "string"},
                        "amount": {"type": "number"},
                    },
                    "required": ["name", "amount"],
                }},
            },
            "required": ["vendor", "date", "currency", "total", "items"],
        },
    },
    "Resume": {
        "description": "Extract candidate information from a CV/resume.",
        "schema": {
            "type": "object",
            "properties": {
                "name":   {"type": ["string", "null"]},
                "email":  {"type": ["string", "null"]},
                "phone":  {"type": ["string", "null"]},
                "skills": {"type": "array", "items": {"type": "string"}},
                "experiences": {"type": "array", "items": {
                    "type": "object",
                    "properties": {
                        "company": {"type": "string"},
                        "role":    {"type": "string"},
                        "years":   {"type": ["string", "null"]},
                    },
                    "required": ["company", "role"],
                }},
            },
            "required": ["name", "email", "skills", "experiences"],
        },
    },
    "Invoice": {
        "description": "Extract billing data from a B2B invoice.",
        "schema": {
            "type": "object",
            "properties": {
                "invoice_number": {"type": ["string", "null"]},
                "seller":         {"type": ["string", "null"]},
                "buyer":          {"type": ["string", "null"]},
                "issue_date":     {"type": ["string", "null"]},
                "due_date":       {"type": ["string", "null"]},
                "subtotal":       {"type": ["number", "null"]},
                "tax":            {"type": ["number", "null"]},
                "total":          {"type": ["number", "null"]},
                "line_items": {"type": "array", "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity":    {"type": "number"},
                        "unit_price":  {"type": "number"},
                        "total":       {"type": "number"},
                    },
                    "required": ["description", "total"],
                }},
            },
            "required": ["invoice_number", "seller", "buyer", "total", "line_items"],
        },
    },
}

SAMPLE_RECEIPT_PATH = Path("assets/sample_receipt.jpg")


def _load_sample_data_uri():
    if not SAMPLE_RECEIPT_PATH.exists():
        return None
    b = SAMPLE_RECEIPT_PATH.read_bytes()
    return f"data:image/jpeg;base64,{base64.b64encode(b).decode()}"


DEFAULT_IMAGE_DATA_URI = _load_sample_data_uri()


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> Vision calls still hit real providers, but rate '
        'limits are lower and 4xx errors are more frequent. Use a production key for '
        'serious extraction work.</div>'
    ))

## 2. The extractor (one call per model)

For document extraction, streaming buys nothing — the JSON is only meaningful once complete. So we use a single sync call per model, in parallel, with up to 2 retries on transient `400`/`429`/`5xx` gateway errors.

Each call asks the model to output JSON matching the chosen schema **by prompt instruction only** (no `response_format` strict mode). This is the most revealing setup: it exposes which models naturally respect a JSON schema, which hallucinate, and which return invalid JSON.

> **⚠ Strict mode vs. prompt-only.** Eden AI also supports `response_format: {type: "json_schema", strict: true}` — the gateway enforces schema conformance server-side. We deliberately don't use it here, because strict mode *forces* every model to produce a schema-conformant object even when the image has nothing to extract — at which point models tend to **hallucinate to fill the schema**. The differences you see below are the model's natural instruction-following on a soft constraint. The one-line switch to strict mode is in the Customize section.

In [ ]:
import asyncio
import time

import aiohttp

MAX_RETRIES = 2  # transient 4xx/5xx from gateway: retry with small backoff


def _build_messages(schema_name, image_url):
    cfg = SCHEMAS[schema_name]
    schema_str = json.dumps(cfg["schema"], indent=2)
    prompt = (
        f"{cfg['description']}\n\n"
        f"Return ONLY a JSON object matching this schema (no prose, no markdown fences):\n"
        f"{schema_str}\n\n"
        "Rules:\n"
        "- If a field is not visible in the document, return null (not the string \"null\").\n"
        "- Do not invent items, line items, or values that are not visible.\n"
        "- Numbers must be JSON numbers, not strings.\n"
    )
    return [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": image_url}},
        ],
    }]


def _strip_fences(text):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t[3:]
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    return t.strip()


# Friendly translations for common schema-violation paths
def _friendly_issue(issue: str) -> str:
    # "$.items[0].name: expected ['string'], got NoneType" -> "Item #1: name missing"
    import re
    m = re.match(r"\$\.([\w_]+)\[(\d+)\]\.([\w_]+): (.*)", issue)
    if m:
        arr, idx, field, rest = m.group(1), int(m.group(2)) + 1, m.group(3), m.group(4)
        arr_label = arr.rstrip("s").replace("_", " ").title()
        if "string \"null\"" in rest:
            return f'{arr_label} #{idx} — "{field}" is the string "null" instead of JSON null'
        if "expected" in rest and "got" in rest:
            return f'{arr_label} #{idx} — "{field}" has wrong type ({rest})'
        return f'{arr_label} #{idx} — "{field}": {rest}'
    m = re.match(r"\$\.([\w_]+): (.*)", issue)
    if m:
        field, rest = m.group(1), m.group(2)
        if "string \"null\"" in rest:
            return f'"{field}" is the string "null" instead of JSON null'
        return f'"{field}" {rest}'
    return issue


def _validate(data, schema):
    """Lightweight schema check. Returns (ok, list_of_issues)."""
    issues = []

    def _check(val, sch, path):
        expected = sch.get("type")
        types = expected if isinstance(expected, list) else [expected]
        type_map = {
            "string": str, "number": (int, float), "integer": int,
            "boolean": bool, "array": list, "object": dict, "null": type(None),
        }
        ok = any(isinstance(val, type_map[t]) for t in types if t in type_map)
        if not ok:
            if val == "null" and "null" in types:
                issues.append(f"{path}: got string \"null\" instead of JSON null")
            else:
                issues.append(f"{path}: expected {types}, got {type(val).__name__}")
            return
        if isinstance(val, dict) and "properties" in sch:
            for k in sch.get("required", []):
                if k not in val:
                    issues.append(f"{path}.{k}: required field missing")
            for k, sub_sch in sch["properties"].items():
                if k in val:
                    _check(val[k], sub_sch, f"{path}.{k}")
        if isinstance(val, list) and "items" in sch:
            for i, item in enumerate(val):
                _check(item, sch["items"], f"{path}[{i}]")

    _check(data, schema, "$")
    return (len(issues) == 0, issues)


async def _call_once(session, payload):
    headers = {
        "Authorization": f"Bearer {EDENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    async with session.post(EDENAI_URL, headers=headers, json=payload,
                            timeout=aiohttp.ClientTimeout(total=90)) as resp:
        body = await resp.text()
        return resp.status, body


async def extract_one(session, model_cfg, schema_name, image_url):
    payload = {
        "model": model_cfg["model"],
        "messages": _build_messages(schema_name, image_url),
    }
    start = time.perf_counter()
    last_err = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            status, body = await _call_once(session, payload)
            if status == 200:
                data = json.loads(body)
                content = data["choices"][0]["message"]["content"]
                break
            # Retry on transient gateway errors only
            last_err = f"HTTP {status}: {body[:200]}"
            if status in (400, 429, 502, 503, 504) and attempt < MAX_RETRIES:
                await asyncio.sleep(0.6 * (attempt + 1))
                continue
            return {
                "label": model_cfg["label"], "latency": time.perf_counter() - start,
                "status": "http_error", "error": last_err,
            }
        except Exception as e:
            last_err = str(e)
            if attempt < MAX_RETRIES:
                await asyncio.sleep(0.6 * (attempt + 1))
                continue
            return {
                "label": model_cfg["label"], "latency": time.perf_counter() - start,
                "status": "network_error", "error": last_err,
            }

    raw = _strip_fences(content)
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as e:
        return {
            "label": model_cfg["label"], "latency": time.perf_counter() - start,
            "status": "invalid_json", "raw": raw[:600], "error": str(e),
        }

    ok, issues = _validate(parsed, SCHEMAS[schema_name]["schema"])
    return {
        "label": model_cfg["label"], "latency": time.perf_counter() - start,
        "status": "ok" if ok else "schema_violation",
        "parsed": parsed,
        "issues": [_friendly_issue(i) for i in issues],
        "raw_issues": issues,
    }

## 3. UI

Pick a schema, drop an image (or paste a URL), press Extract. Each panel shows the JSON output, its validation status, and the latency. The schema selector hot-swaps between Receipt / Resume / Invoice — no code change needed.

In [ ]:
import requests

from ipywidgets import (
    Button, Dropdown, FileUpload, GridBox, HBox, HTML as HTMLWidget,
    Layout, Output, Text, ToggleButtons, VBox,
)
from IPython.display import display

schema_dropdown = Dropdown(
    options=list(SCHEMAS.keys()),
    value="Receipt",
    description="Schema:",
    layout=Layout(width="260px"),
)
schema_preview = HTMLWidget()


def _render_schema_preview(name):
    cfg = SCHEMAS[name]
    props = cfg["schema"].get("properties", {})
    required = set(cfg["schema"].get("required", []))
    rows = ""
    for k, sub in props.items():
        is_req = k in required
        types = sub.get("type")
        if isinstance(types, list):
            type_label = " | ".join(t for t in types if t != "null") + (" or null" if "null" in types else "")
        elif types == "array":
            inner = sub.get("items", {}).get("type", "object")
            type_label = f"array of {inner}"
        else:
            type_label = str(types)
        rows += (
            '<div style="display:flex;gap:10px;font-family:monospace;font-size:11px;padding:1px 0;">'
            f'<span style="width:120px;color:#222;">{k}{" *" if is_req else ""}</span>'
            f'<span style="color:#666;">{type_label}</span>'
            '</div>'
        )
    schema_preview.value = (
        f'<details style="font-family:sans-serif;font-size:12px;background:#f8f9fa;'
        f'padding:6px 10px;border-radius:4px;border-left:3px solid #007bff;">'
        f'<summary style="cursor:pointer;"><b>{name} schema</b> '
        f'<span style="color:#666;">— {cfg["description"]}</span></summary>'
        f'<div style="margin-top:6px;">{rows}'
        f'<div style="margin-top:6px;color:#888;font-size:10px;">* = required</div></div>'
        f'</details>'
    )


_render_schema_preview(schema_dropdown.value)
schema_dropdown.observe(lambda c: _render_schema_preview(c["new"]) if c["name"] == "value" else None, names="value")


# Image input
image_url_box = Text(
    value="",
    placeholder="Paste an image URL (or upload below)",
    layout=Layout(width="100%"),
)
image_upload = FileUpload(accept="image/*", multiple=False, description="📷 Upload")
sample_btn = Button(description="📄 Use sample receipt", layout=Layout(width="200px"))
image_status = HTMLWidget(value='<span style="color:#888;font-family:sans-serif;font-size:11px;">no image loaded</span>')

image_preview = HTMLWidget(value=(
    '<div style="max-width:280px;height:200px;border:2px dashed #ddd;border-radius:4px;'
    'display:flex;align-items:center;justify-content:center;color:#aaa;'
    'font-family:sans-serif;font-size:13px;text-align:center;padding:10px;">'
    'No image loaded.<br>Click <b>Use sample receipt</b><br>or upload one.'
    '</div>'
))

_current_image = {"data_uri": None, "source_url": None, "filename": None}


def _set_image_from_data_uri(data_uri, source_label="image"):
    _current_image["data_uri"] = data_uri
    _current_image["source_url"] = None
    _current_image["filename"] = source_label
    image_preview.value = (
        f'<img src="{data_uri}" '
        'style="max-width:280px;max-height:200px;border:1px solid #ddd;border-radius:4px;" />'
    )
    image_status.value = (
        f'<span style="color:#28a745;font-family:sans-serif;font-size:11px;">'
        f'✓ {source_label}</span>'
    )


def _on_upload_change(change):
    files = change["new"]
    if not files:
        return
    f = files[0] if isinstance(files, (list, tuple)) else next(iter(files.values()))
    content = f["content"] if isinstance(f, dict) else f.content
    mime = (f["type"] if isinstance(f, dict) else f.type) or "image/jpeg"
    name = (f["name"] if isinstance(f, dict) else f.name) or "uploaded image"
    b64 = base64.b64encode(bytes(content)).decode()
    _set_image_from_data_uri(f"data:{mime};base64,{b64}", name)


def _on_url_change(change):
    new_url = (change["new"] or "").strip()
    if not new_url:
        return
    _current_image["data_uri"] = None
    _current_image["source_url"] = new_url
    _current_image["filename"] = new_url[:60]
    image_preview.value = (
        f'<img src="{new_url}" '
        'onerror="this.parentNode.innerHTML=\'<div style=&quot;color:#dc3545;font-family:sans-serif;font-size:12px;padding:20px;&quot;>Failed to load image from URL.</div>\'"'
        'style="max-width:280px;max-height:200px;border:1px solid #ddd;border-radius:4px;" />'
    )
    image_status.value = (
        f'<span style="color:#17a2b8;font-family:sans-serif;font-size:11px;">'
        f'URL set (fetched on Extract)</span>'
    )


def _on_sample(_):
    if DEFAULT_IMAGE_DATA_URI is None:
        image_status.value = (
            '<span style="color:#dc3545;font-family:sans-serif;font-size:11px;">'
            'sample_receipt.jpg missing — re-run setup</span>'
        )
        return
    _set_image_from_data_uri(DEFAULT_IMAGE_DATA_URI, "sample_receipt.jpg")


image_upload.observe(_on_upload_change, names="value")
image_url_box.observe(_on_url_change, names="value")
sample_btn.on_click(_on_sample)


def _resolve_image_url():
    if _current_image["data_uri"]:
        return _current_image["data_uri"]
    url = _current_image["source_url"]
    if not url:
        return None
    r = requests.get(url, headers={"User-Agent": "cookbook/1.0"}, timeout=20)
    r.raise_for_status()
    mime = r.headers.get("Content-Type", "image/jpeg").split(";")[0].strip()
    if not mime.startswith("image/"):
        mime = "image/jpeg"
    b64 = base64.b64encode(r.content).decode()
    return f"data:{mime};base64,{b64}"


# Action buttons
extract_btn = Button(description="📄 Extract", button_style="primary")
clear_btn = Button(description="Clear")
view_toggle = ToggleButtons(
    options=[("JSON panels", "json"), ("Compare table", "table")],
    value="json",
    style={"button_width": "130px"},
)

panel_layout = Layout(border="1px solid #ddd", padding="8px", height="320px", overflow="auto")
panels = [Output(layout=panel_layout) for _ in MODELS]
headers = [HTMLWidget() for _ in MODELS]


def _empty_header(i):
    m = MODELS[i]
    return (
        f'<div style="font-family:sans-serif;font-size:13px;padding:4px;">'
        f'<b>{m["label"]}</b> <span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
    )


def _set_empty_panel(i):
    headers[i].value = _empty_header(i)
    panels[i].clear_output()
    with panels[i]:
        display(HTML(
            '<div style="font-family:sans-serif;color:#aaa;font-size:12px;text-align:center;padding:40px 10px;">'
            'Drop a document and click<br><b>📄 Extract</b><br>to start</div>'
        ))


for i in range(len(MODELS)):
    _set_empty_panel(i)


panel_blocks = [
    VBox([headers[i], panels[i]], layout=Layout(border="1px solid #eee", padding="4px", border_radius="4px"))
    for i in range(len(MODELS))
]
grid = GridBox(
    panel_blocks,
    layout=Layout(grid_template_columns="repeat(2, 1fr)", grid_gap="8px"),
)

table_view = Output(layout=Layout(border="1px solid #eee", padding="8px", border_radius="4px", display="none"))


def _on_view_change(change):
    if change["new"] == "json":
        grid.layout.display = ""
        table_view.layout.display = "none"
    else:
        grid.layout.display = "none"
        table_view.layout.display = ""


view_toggle.observe(_on_view_change, names="value")


image_panel = VBox([
    HTMLWidget(value='<b style="font-family:sans-serif;font-size:13px;">Document</b>'),
    image_preview,
    HBox([sample_btn, image_upload]),
    HBox([HTMLWidget(value='<span style="font-family:sans-serif;font-size:12px;color:#666;width:30px;">URL:</span>'), image_url_box]),
    image_status,
])

summary_out = Output()

display(VBox([
    image_panel,
    HBox([schema_dropdown]),
    schema_preview,
    HBox([extract_btn, clear_btn, view_toggle]),
    grid,
    table_view,
    summary_out,
]))

## 4. Wire it up

Extract → fan out the same image + schema to all four models in parallel, render each result with a status badge: `valid`, `schema violation`, `invalid JSON`, or `error`.

In [ ]:
import nest_asyncio
from IPython.display import HTML, clear_output

nest_asyncio.apply()

STATUS_STYLES = {
    "ok":               ("valid ✓",        "#28a745"),
    "schema_violation": ("schema issue ⚠", "#fd7e14"),
    "invalid_json":     ("invalid JSON ✗", "#dc3545"),
    "http_error":       ("HTTP error ✗",   "#dc3545"),
    "network_error":    ("network error ✗", "#dc3545"),
}

# Animated loading dots via inline CSS keyframes (injected once)
display(HTML('''
<style>
@keyframes cb_blink { 0%, 100% { opacity: 0.2; } 50% { opacity: 1; } }
.cb-dot { animation: cb_blink 1.2s infinite both; display:inline-block; }
.cb-dot:nth-child(2) { animation-delay: 0.2s; }
.cb-dot:nth-child(3) { animation-delay: 0.4s; }
</style>
'''))


def _render_header(i, status_key, latency):
    label = MODELS[i]["label"]
    model = MODELS[i]["model"]
    status_label, color = STATUS_STYLES.get(status_key, (status_key, "#6c757d"))
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'  <div><b style="font-size:13px;">{label}</b> '
        f'<span style="color:#888;font-size:11px;">{model}</span></div>'
        f'  <div><span style="background:{color};color:white;padding:3px 10px;'
        f'border-radius:10px;font-size:11px;font-weight:600;">'
        f'{status_label} · {latency:.2f}s</span></div>'
        '</div>'
    )


def _set_loading_header(i):
    label = MODELS[i]["label"]
    model = MODELS[i]["model"]
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'  <div><b style="font-size:13px;">{label}</b> '
        f'<span style="color:#888;font-size:11px;">{model}</span></div>'
        '  <div><span style="background:#17a2b8;color:white;padding:3px 10px;'
        'border-radius:10px;font-size:11px;font-weight:600;">extracting'
        '<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></span></div>'
        '</div>'
    )


def _copy_button_html(text_to_copy, label="Copy JSON"):
    # Escape backticks and backslashes for the JS template literal
    safe = text_to_copy.replace("\\", "\\\\").replace("`", "\\`").replace("</", "<\\/")
    return (
        f'<button onclick="navigator.clipboard.writeText(`{safe}`);'
        f'this.textContent=&quot;Copied ✓&quot;;setTimeout(()=>this.textContent=&quot;{label}&quot;,1500);" '
        f'style="font-size:11px;padding:2px 8px;border:1px solid #ddd;background:#fff;'
        f'border-radius:3px;cursor:pointer;float:right;">{label}</button>'
    )


def _render_panel(i, result):
    panel = panels[i]
    panel.clear_output()
    status = result["status"]
    _render_header(i, status, result["latency"])

    with panel:
        if status == "ok":
            json_text = json.dumps(result["parsed"], indent=2)
            display(HTML(
                _copy_button_html(json_text) +
                f'<pre style="font-size:11px;line-height:1.4;background:#f8f9fa;padding:8px;'
                f'border-radius:4px;overflow:auto;margin-top:4px;">{json_text}</pre>'
            ))
        elif status == "schema_violation":
            issues_html = "".join(
                f'<li style="color:#fd7e14;">{issue}</li>' for issue in result["issues"][:8]
            )
            json_text = json.dumps(result["parsed"], indent=2)
            display(HTML(
                _copy_button_html(json_text) +
                f'<div style="font-family:sans-serif;font-size:11px;clear:both;">'
                f'<b>Schema issues:</b><ul style="margin:4px 0;padding-left:20px;">{issues_html}</ul></div>'
                f'<pre style="font-size:11px;line-height:1.4;background:#fff3cd;padding:8px;'
                f'border-radius:4px;overflow:auto;">{json_text}</pre>'
            ))
        elif status == "invalid_json":
            raw = result.get("raw", "")
            display(HTML(
                f'<div style="font-family:monospace;font-size:11px;color:#dc3545;">'
                f'JSON parse failed: {result["error"]}</div>'
                + (_copy_button_html(raw, "Copy raw") if raw else "") +
                f'<pre style="font-size:11px;background:#f8d7da;padding:8px;border-radius:4px;'
                f'overflow:auto;clear:both;">{raw}</pre>'
            ))
        else:
            display(HTML(
                f'<div style="font-family:monospace;font-size:11px;color:#dc3545;'
                f'padding:8px;background:#f8d7da;border-radius:4px;">'
                f'{result.get("error", "unknown error")[:400]}</div>'
            ))


def _collect_table_fields(results):
    """Find the union of top-level keys across all successful parses,
    in the order they appear in the schema."""
    schema_name = schema_dropdown.value
    schema_keys = list(SCHEMAS[schema_name]["schema"]["properties"].keys())
    fields = []
    for k in schema_keys:
        for r in results:
            if r.get("parsed") and k in r["parsed"]:
                fields.append(k)
                break
    return fields


def _format_cell_value(v):
    if v is None:
        return '<span style="color:#28a745;font-weight:600;">null</span>'
    if v == "null":
        return '<span style="color:#dc3545;font-weight:600;">"null" (str!)</span>'
    if isinstance(v, list):
        return f'<span style="color:#666;">[{len(v)} items]</span>'
    if isinstance(v, dict):
        return f'<span style="color:#666;">{{{len(v)} keys}}</span>'
    s = str(v)
    if len(s) > 40:
        s = s[:38] + "…"
    return s


def _render_table(results):
    fields = _collect_table_fields(results)
    if not fields:
        with table_view:
            clear_output()
            display(HTML('<div style="font-family:sans-serif;color:#888;font-size:12px;">No data to compare yet.</div>'))
        return

    # Per row, detect disagreement (≥2 unique non-error values)
    def values_disagree(field):
        vals = []
        for r in results:
            if r.get("parsed") and field in r["parsed"]:
                v = r["parsed"][field]
                if isinstance(v, (list, dict)):
                    vals.append(json.dumps(v, sort_keys=True))
                else:
                    vals.append(repr(v))
        return len(set(vals)) > 1

    head = '<th style="text-align:left;padding:6px 10px;background:#f8f9fa;border-bottom:2px solid #ddd;">Field</th>'
    for r in results:
        head += (
            f'<th style="text-align:left;padding:6px 10px;background:#f8f9fa;'
            f'border-bottom:2px solid #ddd;">{r["label"]}</th>'
        )

    rows_html = ""
    for field in fields:
        disagree = values_disagree(field)
        bg = "#fff8e1" if disagree else "white"
        row = f'<tr style="background:{bg};">'
        row += f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:monospace;font-size:11px;"><b>{field}</b></td>'
        for r in results:
            if r["status"] in ("http_error", "network_error", "invalid_json"):
                cell = '<span style="color:#dc3545;">—</span>'
            elif r.get("parsed") and field in r["parsed"]:
                cell = _format_cell_value(r["parsed"][field])
            else:
                cell = '<span style="color:#aaa;">—</span>'
            row += f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:monospace;font-size:11px;">{cell}</td>'
        row += '</tr>'
        rows_html += row

    # Status row
    status_row = '<tr><td style="padding:6px 10px;border-top:2px solid #ddd;font-family:sans-serif;font-size:11px;color:#666;"><b>status</b></td>'
    for r in results:
        label, color = STATUS_STYLES.get(r["status"], (r["status"], "#6c757d"))
        status_row += (
            f'<td style="padding:6px 10px;border-top:2px solid #ddd;">'
            f'<span style="background:{color};color:white;padding:2px 8px;border-radius:8px;'
            f'font-size:10px;font-weight:600;">{label} · {r["latency"]:.2f}s</span></td>'
        )
    status_row += '</tr>'

    legend = (
        '<div style="font-family:sans-serif;font-size:11px;color:#666;margin-bottom:8px;">'
        '<span style="background:#fff8e1;padding:2px 6px;border-radius:3px;">yellow row</span>'
        ' = models disagree on this field · '
        '<span style="color:#28a745;font-weight:600;">null</span> = JSON null · '
        '<span style="color:#dc3545;font-weight:600;">"null"</span> = string instead of null</div>'
    )

    with table_view:
        clear_output()
        display(HTML(
            legend +
            '<table style="border-collapse:collapse;width:100%;font-family:sans-serif;">'
            f'<thead><tr>{head}</tr></thead>'
            f'<tbody>{rows_html}{status_row}</tbody>'
            '</table>'
        ))


def _render_summary(results):
    fastest = min(results, key=lambda r: r["latency"])
    valid_models = [r["label"] for r in results if r["status"] == "ok"]
    issues_models = [r["label"] for r in results if r["status"] == "schema_violation"]
    failed_models = [r["label"] for r in results if r["status"] in ("invalid_json", "http_error", "network_error")]
    parts = []
    if valid_models:
        parts.append(f'<span style="color:#28a745;">✓ Valid: {", ".join(valid_models)}</span>')
    if issues_models:
        parts.append(f'<span style="color:#fd7e14;">⚠ Schema issues: {", ".join(issues_models)}</span>')
    if failed_models:
        parts.append(f'<span style="color:#dc3545;">✗ Failed: {", ".join(failed_models)}</span>')
    parts.append(f'<span style="color:#666;">⚡ Fastest: <b>{fastest["label"]}</b> ({fastest["latency"]:.2f}s)</span>')
    with summary_out:
        clear_output()
        display(HTML(
            f'<div style="background:#f8f9fa;padding:10px 12px;border-radius:4px;'
            f'border-left:4px solid #007bff;font-family:sans-serif;font-size:13px;'
            f'display:flex;gap:18px;flex-wrap:wrap;">{"".join(parts)}</div>'
        ))


async def run_extraction(schema_name, image_url):
    for i in range(len(MODELS)):
        panels[i].clear_output()
        with panels[i]:
            display(HTML(
                '<div style="font-family:sans-serif;color:#17a2b8;font-size:13px;text-align:center;padding:40px 10px;">'
                'extracting<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></div>'
            ))
        _set_loading_header(i)
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[
            extract_one(session, MODELS[i], schema_name, image_url)
            for i in range(len(MODELS))
        ])
    for i, r in enumerate(results):
        _render_panel(i, r)
    _render_table(results)
    _render_summary(results)
    return results


def on_extract(_):
    if _current_image["data_uri"] is None and not _current_image["source_url"]:
        with summary_out:
            clear_output()
            display(HTML(
                '<div style="color:#dc3545;font-family:sans-serif;font-size:12px;padding:8px;">'
                'No image loaded. Click <b>📄 Use sample receipt</b> or upload one.</div>'
            ))
        return
    try:
        image_url = _resolve_image_url()
    except Exception as e:
        with summary_out:
            clear_output()
            display(HTML(
                f'<div style="color:#dc3545;font-family:monospace;font-size:12px;padding:8px;">'
                f'Could not load image: {e}</div>'
            ))
        return
    if not image_url:
        return
    asyncio.run(run_extraction(schema_dropdown.value, image_url))


def on_clear(_):
    for i in range(len(MODELS)):
        _set_empty_panel(i)
    table_view.clear_output()
    summary_out.clear_output()


extract_btn.on_click(on_extract)
clear_btn.on_click(on_clear)

## 5. Try these documents

Documents that reveal different failure modes:

- **A real grocery receipt** → tests OCR + arithmetic. Does the model sum the items, or invent a total?
- **A handwritten note** → tests vision robustness. Does the model say it can't read, or hallucinate plausible text?
- **A non-document image** (try the default Unsplash URL) → tests refusal. Does the model say "no document here" or invent a fake receipt? *Spoiler: at least one model will lie.*
- **A multi-page PDF screenshot** → tests visual reasoning. Does it pick the right invoice fields?
- **A document in a foreign language** → tests multilingual OCR + extraction in one shot.

## 6. Customize

**Bring your own schema.** Add an entry to `SCHEMAS`:
```python
SCHEMAS["ID Card"] = {
    "description": "Extract identity fields from an ID card or passport.",
    "schema": {
        "type": "object",
        "properties": {
            "full_name":     {"type": ["string", "null"]},
            "date_of_birth": {"type": ["string", "null"]},
            "id_number":     {"type": ["string", "null"]},
            "expiry_date":   {"type": ["string", "null"]},
        },
        "required": ["full_name", "id_number"],
    },
}
```
The dropdown picks it up on the next cell rerun.

**Switch to strict mode.** For production, pass the schema directly in `response_format` to force valid JSON server-side (requires `supports_response_schema: true`, which all four default models have):
```python
payload["response_format"] = {
    "type": "json_schema",
    "json_schema": {"name": "extraction", "schema": SCHEMAS[schema_name]["schema"], "strict": True},
}
```
OpenAI's strict mode requires `additionalProperties: false` on every nested object — easy to add programmatically with a tiny helper. Anthropic is more permissive.

**Batch mode.** Loop over a folder of receipts and write a CSV:
```python
for path in pathlib.Path("receipts/").glob("*.jpg"):
    b64 = base64.b64encode(path.read_bytes()).decode()
    result = await extract_one(session, MODELS[0], "Receipt", f"data:image/jpeg;base64,{b64}")
    # ... append to CSV
```
Turns this notebook into a tiny eval harness for picking the best provider on YOUR document distribution.